<div style='background:#0d1117;padding:30px 24px;border-radius:10px;text-align:center;border-left:6px solid #2196F3'>
<h1 style='color:#ffffff;margin:0;font-size:2em'>Person 5 — Relationship Charts</h1>
<p style='color:#a8dadc;font-size:1.1em;margin:10px 0 4px'>Scatter: Age vs Rating &nbsp;|&nbsp; Bubble: Age vs Potential (Size = Value)</p>
<p style='color:#a8dadc;font-size:0.95em;margin:0'>By: Mohamed Galal</p>
</div>

In [1]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('cleaned_data.csv')
df = df.sample(5000, random_state=42)
print(f'Dataset loaded: {df.shape[0]:,} players')
df.head(3)

Dataset loaded: 5,000 players


,short_name,player_positions,overall,potential,value_eur,wage_eur,age,club_name,nationality_name,position_group
1772135,S. Dooley,"LM, RM",63,63,475000.0,1000.0,28,Rochdale,Northern Ireland,Midfielder
5164960,M. Vilotić,CB,70,70,1300000.0,15000.0,30,Grasshopper,Serbia,Defender
3334931,R. Fernández,"CAM, CM",70,70,900000.0,5000.0,34,O'Higgins,Argentina,Midfielder


In [2]:
df.columns

Index(['short_name', 'player_positions', 'overall', 'potential', 'value_eur',
       'wage_eur', 'age', 'club_name', 'nationality_name', 'position_group'],
      dtype='str')

In [3]:
top_clubs     = sorted(df['club_name'].value_counts().head(20).index.tolist())
top_countries = sorted(df['nationality_name'].value_counts().head(20).index.tolist())
print('Clubs:    ', top_clubs[:5], '...')
print('Countries:', top_countries[:5], '...')

Clubs:     ['Atlético Madrid', 'Benfica', 'Celta de Vigo', 'Coventry City', 'Eintracht Frankfurt'] ...
Countries: ['Argentina', 'Brazil', 'Chile', 'Colombia', 'England'] ...


---
## Chart 1 — Scatter: Age vs Overall Rating


In [4]:
# Widgets
s_filter = widgets.Dropdown(options=['Club', 'Country'], value='Club', description='Filter by:')
s_select = widgets.Dropdown(options=top_clubs, value=top_clubs[0], description='Select:')
s_age    = widgets.IntRangeSlider(value=[20, 35], min=16, max=45, step=1,
                                   description='Age range:', continuous_update=False,
                                   style={'description_width': 'initial'},
                                   layout=widgets.Layout(width='55%'))
s_out    = widgets.Output()

def on_s_filter_change(change):
    if change['new'] == 'Club':
        s_select.options = top_clubs;     s_select.value = top_clubs[0]
    else:
        s_select.options = top_countries; s_select.value = top_countries[0]

s_filter.observe(on_s_filter_change, names='value')

def update_scatter(*args):
    if s_filter.value == 'Club':
        temp     = df[df['club_name'] == s_select.value].copy()
        subtitle = f'Club: {s_select.value}'
    else:
        temp     = df[df['nationality_name'] == s_select.value].copy()
        subtitle = f'Country: {s_select.value}'

    temp = temp[(temp['age'] >= s_age.value[0]) & (temp['age'] <= s_age.value[1])].copy()

    s_out.clear_output(wait=True)
    with s_out:
        if temp.empty:
            print('No data for this selection.')
            return

        threshold = temp['overall'].quantile(0.95)
        temp['Player Type'] = temp['overall'].apply(
            lambda x: 'Top Rated (Outlier)' if x >= threshold else 'Player'
        )
        temp['label'] = temp.apply(
            lambda r: r['short_name'] if r['Player Type'] == 'Top Rated (Outlier)' else '', axis=1
        )

        fig = px.scatter(
            temp,
            x='age', y='overall',
            color='Player Type',
            color_discrete_map={
                'Player':             'lightblue',
                'Top Rated (Outlier)':'lightcoral'
            },
            text='label',
            hover_name='short_name',
            title=f'<b>Age vs Overall Rating</b><br><sup>{subtitle} | Age {s_age.value[0]}\u2013{s_age.value[1]}</sup>',
            labels={'age': 'Age', 'overall': 'Overall Rating', 'Player Type': 'Player Type'}
        )
        fig.update_traces(
            marker=dict(line=dict(width=1, color='black')),
            textposition='top center', textfont=dict(size=9)
        )
        fig.update_layout(
            title_x=0.5,
            legend=dict(x=0.98, y=0.98, xanchor='right', yanchor='top',
                        bgcolor='white', bordercolor='black', borderwidth=1),
            xaxis=dict(showgrid=True, gridcolor='lightgrey',
                       showline=True, linecolor='black', mirror=True),
            yaxis=dict(range=[0, temp['overall'].max() + 5],
                       showgrid=True, gridcolor='lightgrey',
                       showline=True, linecolor='black', mirror=True),
            shapes=[dict(type='rect', xref='paper', yref='paper',
                         x0=0, y0=0, x1=1, y1=1,
                         line=dict(color='black', width=2))],
            plot_bgcolor='white', paper_bgcolor='white',
            width=850, height=520
        )
        fig.show()

s_filter.observe(lambda c: update_scatter(), 'value')
s_select.observe(lambda c: update_scatter(), 'value')
s_age.observe(lambda c: update_scatter(),    'value')

display(widgets.VBox([s_filter, s_select, s_age, s_out]))
update_scatter()

---
## Chart 2 — Bubble: Age vs Potential  (Size = Market Value)


In [5]:
# Widgets
b_filter = widgets.Dropdown(options=['Club', 'Country'], value='Club', description='Filter by:')
b_select = widgets.Dropdown(options=top_clubs, value=top_clubs[0], description='Select:')
b_age    = widgets.IntRangeSlider(value=[20, 35], min=16, max=45, step=1,
                                   description='Age range:', continuous_update=False,
                                   style={'description_width': 'initial'},
                                   layout=widgets.Layout(width='55%'))
b_out    = widgets.Output()

def on_b_filter_change(change):
    if change['new'] == 'Club':
        b_select.options = top_clubs;     b_select.value = top_clubs[0]
    else:
        b_select.options = top_countries; b_select.value = top_countries[0]

b_filter.observe(on_b_filter_change, names='value')

def update_bubble(*args):
    if b_filter.value == 'Club':
        temp     = df[df['club_name'] == b_select.value].copy()
        subtitle = f'Club: {b_select.value}'
    else:
        temp     = df[df['nationality_name'] == b_select.value].copy()
        subtitle = f'Country: {b_select.value}'

    temp = temp[(temp['age'] >= b_age.value[0]) & (temp['age'] <= b_age.value[1])]
    temp = temp.dropna(subset=['value_eur', 'potential'])
    temp = temp[temp['value_eur'] > 0].copy()

    b_out.clear_output(wait=True)
    with b_out:
        if temp.empty:
            print('No data for this selection.')
            return

        threshold = temp['potential'].quantile(0.95)
        temp['Player Type'] = temp['potential'].apply(
            lambda x: 'High Potential (Outlier)' if x >= threshold else 'Player'
        )
        temp['label'] = temp.apply(
            lambda r: r['short_name'] if r['Player Type'] == 'High Potential (Outlier)' else '', axis=1
        )

        fig = px.scatter(
            temp,
            x='age', y='potential',
            size='value_eur', size_max=50,
            color='Player Type',
            color_discrete_map={
                'Player':                  'lightblue',
                'High Potential (Outlier)':'lightgreen'
            },
            text='label',
            hover_name='short_name',
            title=f'<b>Age vs Potential</b>  <sup>(Bubble Size = Market Value \u20ac)</sup><br><sup>{subtitle} | Age {b_age.value[0]}\u2013{b_age.value[1]}</sup>',
            labels={'age': 'Age', 'potential': 'Potential', 'value_eur': 'Value (\u20ac)', 'Player Type': 'Player Type'}
        )
        fig.update_traces(
            marker=dict(line=dict(width=1, color='black')),
            textposition='top center', textfont=dict(size=9)
        )
        fig.update_layout(
            title_x=0.5,
            legend=dict(x=0.98, y=0.98, xanchor='right', yanchor='top',
                        bgcolor='white', bordercolor='black', borderwidth=1),
            xaxis=dict(showgrid=True, gridcolor='lightgrey',
                       showline=True, linecolor='black', mirror=True),
            yaxis=dict(range=[0, temp['potential'].max() + 5],
                       showgrid=True, gridcolor='lightgrey',
                       showline=True, linecolor='black', mirror=True),
            shapes=[dict(type='rect', xref='paper', yref='paper',
                         x0=0, y0=0, x1=1, y1=1,
                         line=dict(color='black', width=2))],
            plot_bgcolor='white', paper_bgcolor='white',
            width=850, height=550
        )
        fig.show()

b_filter.observe(lambda c: update_bubble(), 'value')
b_select.observe(lambda c: update_bubble(), 'value')
b_age.observe(lambda c: update_bubble(),    'value')

display(widgets.VBox([b_filter, b_select, b_age, b_out]))
update_bubble()

---
## Chart 3 — Column Chart : Top 10 Clubs by Average Player Rating

In [6]:
top_clubs = (
    df.groupby("club_name")["overall"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_clubs,
    x="club_name",
    y="overall",
    title="Top 10 Clubs by Average Player Rating",
    labels={
        "overall": "Average Rating",
        "club_name": "Club"
    },
    text="overall"
)

fig.update_traces(
    texttemplate='%{text:.1f}',
    textposition='outside'
)

fig.update_layout(
    title_x=0.5,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()

---
## Chart 4 — Bar Chart : Top 10 Players

In [7]:
top_players = df.sort_values("overall", ascending=False).head(10)

fig = px.bar(
    top_players,
    x="short_name",
    y="overall",
    color="overall",
    title="Top 10 Players in FIFA",
    labels={
        "short_name": "Player",
        "overall": "Rating"
    },
    text="overall",
    color_continuous_scale="viridis"
)

fig.update_traces(
    texttemplate='%{text}',
    textposition='outside'
)

fig.update_layout(
    title_x=0.5,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    coloraxis_showscale=False
)

fig.show()

--- 
## Chart 5 — Box Plot : Salary Distribution by Position

In [8]:
# Some players have multiple positions → take only the first one
df["main_position"] = df["player_positions"].str.split(",").str[0]

fig = px.box(
    df,
    x="main_position",
    y="wage_eur",
    title="Salary Distribution by Player Position",
    labels={
        "main_position": "Position",
        "wage_eur": "Wage (EUR)"
    },
    color="main_position"
)

fig.update_layout(
    title_x=0.5,
    xaxis_tickangle=-45,
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False
)

fig.show()